# PEMDT-Cirrhosis — Latent Staging v2 (daily windows · K-sweep · per-treatment fusion)

Tier-1 improvements over v1, each tied to a problem we diagnosed:

| Fix | Why |
|---|---|
| **Daily windows** (was 60-min) | hourly transitions were ~0.99 on the diagonal — no dynamics to model. Daily aggregation makes transitions learnable. |
| **K-sweep (3–6) + restarts** | K=4 over-segmented the middle (co-severe stages 1/2). Select K by held-out predictive LL **and** mortality monotonicity. |
| **Per-treatment fusion** | the "any treatment" covariate was 68% prevalent → no contrast. Test lactulose / vasopressor / albumin / paracentesis **separately**. |
| **Richer + fixed observations** | add Ammonia, Lactate; optional GCS re-extraction (sum of components — the v1 GCS was mis-averaged). |

**Headline experiment unchanged:** for each treatment, compare `baseline` / `emission`
(treatment = observation of state) / `transition` (treatment = driver of dynamics) by a fair
one-step-ahead predictive likelihood of next-day labs.

**Honest caveats:** the fusion comparison is *associational*, not causal (confounding by
indication). Higher K fits better but is less ordinal — we prefer the most interpretable K
with monotonic outcomes. Presented for review, not clinical use.

**Inputs:** `cirrhosis_timeline.csv`, `cirrhosis_cohort.csv`. Treatment + GCS steps read MIMIC-IV.


## HMM library

In [1]:
"""Missing-aware diagonal-Gaussian HMM with optional treatment fusion.
Modes:
  baseline   : Gaussian emissions, homogeneous transitions
  emission   : Gaussian + Bernoulli(treatment) emissions, homogeneous transitions
  transition : Gaussian emissions, treatment-conditioned transitions (A_off / A_on)
"""
import numpy as np

VAR_FLOOR = 1e-3

def emission_logB(O, mu, var, u=None, p_treat=None):
    """O: T x D (NaN allowed). returns logB: T x K. Optional Bernoulli treatment emission."""
    T, D = O.shape; K = mu.shape[0]
    mask = ~np.isnan(O)                                  # T x D
    diff = O[:, None, :] - mu[None, :, :]                # T x K x D
    term = -0.5 * (np.log(2*np.pi*var)[None] + (diff**2)/var[None])
    term = np.where(mask[:, None, :], term, 0.0)
    logB = term.sum(axis=2)                              # T x K
    if u is not None and p_treat is not None:
        pt = np.clip(p_treat, 1e-4, 1-1e-4)              # K
        logB = logB + (u[:, None]*np.log(pt)[None] + (1-u[:, None])*np.log(1-pt)[None])
    return logB

def fwd_bwd(logB, pi, A_t):
    """A_t: (T-1) x K x K time-indexed transition matrices. Scaled forward-backward."""
    T, K = logB.shape
    m = logB.max(axis=1); B = np.exp(logB - m[:, None])
    alpha = np.zeros((T, K)); c = np.zeros(T)
    alpha[0] = pi * B[0]; c[0] = alpha[0].sum() + 1e-300; alpha[0] /= c[0]
    for t in range(1, T):
        alpha[t] = (alpha[t-1] @ A_t[t-1]) * B[t]
        c[t] = alpha[t].sum() + 1e-300; alpha[t] /= c[t]
    beta = np.zeros((T, K)); beta[-1] = 1.0
    for t in range(T-2, -1, -1):
        beta[t] = (A_t[t] @ (B[t+1]*beta[t+1])) / c[t+1]
    gamma = alpha*beta; gamma /= gamma.sum(axis=1, keepdims=True) + 1e-300
    loglik = np.sum(np.log(c)) + np.sum(m)
    return alpha, beta, gamma, B, c, loglik

def _init(seqs, K, seed=0):
    rng = np.random.default_rng(seed)
    allobs = np.vstack([s["O"] for s in seqs])
    D = allobs.shape[1]
    med = np.nanmedian(allobs, axis=0)
    # severity proxy = mean of standardized obs (already standardized upstream) -> quantile init
    sev = np.nanmean(allobs, axis=1)
    qs = np.nanquantile(sev, np.linspace(0.1, 0.9, K))
    mu = np.tile(med, (K, 1)).astype(float)
    # spread state means along each dim by quantiles of that dim
    for d in range(D):
        col = allobs[:, d]; col = col[~np.isnan(col)]
        if len(col):
            mu[:, d] = np.quantile(col, np.linspace(0.2, 0.8, K))
    var = np.tile(np.nanvar(allobs, axis=0), (K, 1)); var[var < VAR_FLOOR] = 1.0
    pi = np.full(K, 1.0/K)
    A = np.full((K, K), 0.1/(K-1)) + np.eye(K)*0.9
    A /= A.sum(axis=1, keepdims=True)
    return pi, A, mu, var

def fit(seqs, K=4, mode="baseline", n_iter=25, seed=0, verbose=False):
    """seqs: list of dict with 'O' (T x D) and optional 'u' (T,) treatment binary."""
    pi, A, mu, var = _init(seqs, K, seed)
    A_on = A.copy(); A_off = A.copy()
    p_treat = np.linspace(0.2, 0.8, K) if mode == "emission" else None
    D = mu.shape[1]
    prev_ll = -np.inf
    for it in range(n_iter):
        pi_acc = np.zeros(K)
        num_mu = np.zeros((K, D)); den_mu = np.zeros((K, D)); num_var = np.zeros((K, D))
        if mode == "transition":
            xi_off = np.zeros((K, K)); xi_on = np.zeros((K, K))
        else:
            xi_sum = np.zeros((K, K))
        tp_num = np.zeros(K); tp_den = np.zeros(K)
        total_ll = 0.0
        for s in seqs:
            O = s["O"]; T = O.shape[0]
            u = s.get("u") if mode != "baseline" else None
            logB = emission_logB(O, mu, var, u=(u if mode == "emission" else None),
                                 p_treat=(p_treat if mode == "emission" else None))
            if mode == "transition" and u is not None:
                A_t = np.stack([A_on if u[t] > 0.5 else A_off for t in range(T-1)]) if T > 1 \
                      else np.zeros((0, K, K))
            else:
                A_t = np.stack([A]*(T-1)) if T > 1 else np.zeros((0, K, K))
            alpha, beta, gamma, B, c, ll = fwd_bwd(logB, pi, A_t)
            total_ll += ll
            pi_acc += gamma[0]
            mask = ~np.isnan(O)
            # mean AND variance from the SAME posteriors (cached gamma)
            Osq = np.where(mask, O*O, 0.0)
            for k in range(K):
                g = gamma[:, k]
                num_mu[k]  += np.nansum(np.where(mask, O, 0.0) * g[:, None], axis=0)
                den_mu[k]  += (mask * g[:, None]).sum(axis=0)
                num_var[k] += (Osq * g[:, None]).sum(axis=0)   # sum gamma*x^2 (suff. stat)
            for t in range(T-1):
                At = (A_on if (mode == "transition" and u is not None and u[t] > 0.5)
                      else A_off if mode == "transition" else A)
                xi = (alpha[t][:, None] * At * (B[t+1]*beta[t+1])[None, :]) / (c[t+1])
                xi /= xi.sum() + 1e-300
                if mode == "transition" and u is not None:
                    if u[t] > 0.5: xi_on += xi
                    else: xi_off += xi
                else:
                    xi_sum += xi
            if mode == "emission" and u is not None:
                tp_num += (gamma * u[:, None]).sum(axis=0)
                tp_den += gamma.sum(axis=0)
        # M-step
        N = len(seqs)
        pi = pi_acc / N; pi /= pi.sum()
        mu = num_mu / np.maximum(den_mu, 1e-6)
        var = num_var / np.maximum(den_mu, 1e-6) - mu**2   # E[x^2]-E[x]^2
        var[var < VAR_FLOOR] = VAR_FLOOR
        if mode == "transition":
            A_off = xi_off / np.maximum(xi_off.sum(axis=1, keepdims=True), 1e-12)
            A_on  = xi_on  / np.maximum(xi_on.sum(axis=1, keepdims=True), 1e-12)
        else:
            A = xi_sum / np.maximum(xi_sum.sum(axis=1, keepdims=True), 1e-12)
        if mode == "emission":
            p_treat = np.clip(tp_num/np.maximum(tp_den, 1e-6), 1e-4, 1-1e-4)
        if verbose:
            print(f"  iter {it+1}: loglik={total_ll:.1f}")
        if abs(total_ll - prev_ll) < 1e-3*abs(prev_ll+1e-9):
            break
        prev_ll = total_ll
    params = dict(pi=pi, mu=mu, var=var, K=K, mode=mode)
    if mode == "transition": params["A_off"], params["A_on"] = A_off, A_on
    else: params["A"] = A
    if mode == "emission": params["p_treat"] = p_treat
    params["loglik"] = total_ll
    return params

def reorder_by_severity(params, direction):
    """direction: array length D, sign of severity per obs dim. Sort states ascending severity."""
    mu = params["mu"]; sev = (mu * direction[None, :]).sum(axis=1)
    order = np.argsort(sev)
    inv = np.argsort(order)
    p = dict(params)
    p["mu"] = mu[order]; p["var"] = params["var"][order]; p["pi"] = params["pi"][order]
    p["pi"] /= p["pi"].sum()
    if params["mode"] == "transition":
        p["A_off"] = params["A_off"][np.ix_(order, order)]
        p["A_on"]  = params["A_on"][np.ix_(order, order)]
    else:
        p["A"] = params["A"][np.ix_(order, order)]
    if "p_treat" in params: p["p_treat"] = params["p_treat"][order]
    p["_order"] = order
    return p

def posterior_stage(params, O, u=None):
    """Return gamma (T x K) and expected stage per window."""
    mode = params["mode"]; T = O.shape[0]
    logB = emission_logB(O, params["mu"], params["var"],
                         u=(u if mode == "emission" else None),
                         p_treat=(params.get("p_treat") if mode == "emission" else None))
    if mode == "transition" and u is not None:
        A_t = np.stack([params["A_on"] if u[t] > 0.5 else params["A_off"] for t in range(T-1)]) if T>1 else np.zeros((0,params["K"],params["K"]))
    else:
        A = params.get("A", params.get("A_off"))
        A_t = np.stack([A]*(T-1)) if T>1 else np.zeros((0,params["K"],params["K"]))
    _,_,gamma,_,_,ll = fwd_bwd(logB, params["pi"], A_t)
    stages = (gamma * np.arange(params["K"])[None, :]).sum(axis=1)
    return gamma, stages, ll

def obs_pred_loglik(params, seqs):
    """FAIR cross-mode metric: one-step-ahead predictive log-lik of the next
    observation vector, P(O_{t+1} | O_{1:t}, u_{1:t}). Gaussian (lab) part only,
    so emission/transition/baseline are scored on the SAME target."""
    mode = params["mode"]; K = params["K"]; mu = params["mu"]; var = params["var"]
    A_def = params.get("A", params.get("A_off"))
    tot = 0.0; nwin = 0
    for s in seqs:
        O = s["O"]; T = O.shape[0]; u = s.get("u")
        logB = emission_logB(O, mu, var,
                             u=(u if mode == "emission" else None),
                             p_treat=(params.get("p_treat") if mode == "emission" else None))
        m = logB.max(axis=1); B = np.exp(logB - m[:, None])      # full emission for filtering
        gaussB = np.exp(emission_logB(O, mu, var) -              # gaussian-only for prediction
                        emission_logB(O, mu, var).max(axis=1)[:, None])
        gm = emission_logB(O, mu, var).max(axis=1)
        alpha = params["pi"] * B[0]; alpha /= alpha.sum() + 1e-300
        for t in range(T-1):
            if mode == "transition" and u is not None:
                A = params["A_on"] if u[t] > 0.5 else params["A_off"]
            else:
                A = A_def
            pred = alpha @ A
            pdens = (pred * gaussB[t+1]).sum()                   # predictive density (scaled)
            tot += np.log(pdens + 1e-300) + gm[t+1]              # undo gauss scaling
            nwin += 1
            alpha = pred * B[t+1]; alpha /= alpha.sum() + 1e-300
    return tot, nwin, tot/max(nwin, 1)

def heldout_loglik(params, seqs):
    tot = 0.0; nwin = 0
    for s in seqs:
        u = s.get("u")
        _,_,ll = posterior_stage(params, s["O"], u=u)
        tot += ll; nwin += s["O"].shape[0]
    return tot, nwin, tot/max(nwin,1)


## Config

In [2]:
# ============================================================
# CONFIG
# ============================================================
import os, json, warnings, time
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

DATA_DIR     = Path("data")
TIMELINE_CSV = DATA_DIR / "cirrhosis_timeline.csv"
COHORT_CSV   = DATA_DIR / "cirrhosis_cohort.csv"
OUT = Path("results/staging_v2_output"); OUT.mkdir(exist_ok=True)
MIMIC_ROOT   = "/Users/sumiyasayeed/Documents/mimic4/mimiciv"

WINDOW       = "1D"                  # daily windows
OBS_FEATURES = ["Bilirubin","INR","Albumin","Creatinine","Sodium","MAP","Ammonia","Lactate"]
K_GRID       = [3, 4, 5, 6]
K_RESTARTS   = 3
N_ITER       = 25
MAX_PATIENTS_FIT = None              # e.g. 3000 for a fast run
RANDOM_SEED  = 1
USE_GCS_FIX  = True                  # re-extract GCS from MIMIC chartevents (sum components)

# Per-class treatments (each tested separately in the fusion experiment)
DRUG_REGEX = {
    "lactulose": r"lactulose|rifaximin",
    "diuretic":  r"furosemide|spironolactone|bumetanide|torsemide",
}
INPUT_ITEMIDS = {
    "vasopressor": [221906,221289,221749,222315,221662,221653],
    "albumin":     [220864,220862],
}
PROC_REGEX = {"paracentesis": r"paracentesis"}    # icu/procedureevents -> d_items
GCS_COMPONENT_IDS = [220739, 223900, 223901]      # Eye, Verbal, Motor
print("Config loaded. window =", WINDOW, "| K_GRID =", K_GRID)


Config loaded. window = 1D | K_GRID = [3, 4, 5, 6]


## Step 1 — Load + daily resample

In [3]:
# ============================================================
# STEP 1 — LOAD + DAILY RESAMPLE  (median/day, ffill within patient)
# ============================================================
co = pd.read_csv(COHORT_CSV, usecols=["subject_id","hadm_id","patient_id",
                                      "is_deceased","decompensated","admittime"])
death  = dict(zip(co.patient_id, co.is_deceased))
decomp = dict(zip(co.patient_id, co.decompensated))

df = pd.read_csv(TIMELINE_CSV, usecols=["patient_id","timestamp"]+OBS_FEATURES+["MELD_Na"],
                 low_memory=False)
if MAX_PATIENTS_FIT:
    keep = pd.Series(df.patient_id.unique()).sample(
        min(MAX_PATIENTS_FIT, df.patient_id.nunique()), random_state=RANDOM_SEED)
    df = df[df.patient_id.isin(set(keep))]
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df["day"] = df["timestamp"].dt.floor(WINDOW)
daily = (df.groupby(["patient_id","day"])[OBS_FEATURES+["MELD_Na"]].median()
           .reset_index().sort_values(["patient_id","day"]))
daily[OBS_FEATURES] = daily.groupby("patient_id")[OBS_FEATURES].ffill()
print(f"✅ daily windows: {len(daily):,} | patients: {daily.patient_id.nunique():,} | "
      f"avg windows/patient: {len(daily)/daily.patient_id.nunique():.1f}")


✅ daily windows: 75,616 | patients: 7,869 | avg windows/patient: 9.6


## Step 1b — GCS re-extraction (optional)

In [4]:
# ============================================================
# STEP 1b (OPTIONAL) — GCS re-extraction (SUM of components), daily
# Fixes the v1 bug where GCS components were averaged, not summed.
# ============================================================
def _csv(d, n):
    for e in (".csv",".csv.gz"):
        if os.path.exists(os.path.join(d, n+e)): return os.path.join(d, n+e)
    return None

def add_gcs(daily, co):
    icu = os.path.join(MIMIC_ROOT, "icu")
    path = _csv(icu, "chartevents")
    if not path:
        print("  chartevents not found — skipping GCS fix."); return daily, OBS_FEATURES
    hadm_pid = dict(zip(co.hadm_id, co.patient_id))
    hadm_set = set(co.hadm_id.dropna().astype(int))
    keep = []
    for ch in pd.read_csv(path, usecols=["hadm_id","charttime","itemid","valuenum"],
                          chunksize=1_000_000, low_memory=False):
        ch = ch[ch.hadm_id.isin(hadm_set) & ch.itemid.isin(GCS_COMPONENT_IDS) & ch.valuenum.notna()]
        if not ch.empty: keep.append(ch)
    if not keep:
        print("  no GCS rows — skipping."); return daily, OBS_FEATURES
    g = pd.concat(keep, ignore_index=True)
    g["charttime"] = pd.to_datetime(g["charttime"], errors="coerce")
    # sum the 3 components at matched charttime (need all three)
    piv = g.pivot_table(index=["hadm_id","charttime"], columns="itemid",
                        values="valuenum", aggfunc="mean")
    piv = piv.dropna(how="any")
    piv["GCS_fixed"] = piv.sum(axis=1).clip(3, 15)
    piv = piv.reset_index()
    piv["patient_id"] = piv["hadm_id"].map(hadm_pid)
    piv["day"] = piv["charttime"].dt.floor(WINDOW)
    gd = piv.groupby(["patient_id","day"])["GCS_fixed"].median().reset_index()
    out = daily.merge(gd, on=["patient_id","day"], how="left")
    out["GCS_fixed"] = out.groupby("patient_id")["GCS_fixed"].ffill()
    obs = OBS_FEATURES + ["GCS_fixed"]
    print(f"✅ GCS_fixed merged ({out['GCS_fixed'].notna().mean()*100:.0f}% of windows)")
    return out, obs

if USE_GCS_FIX and os.path.exists(os.path.join(MIMIC_ROOT,"icu")):
    daily, OBS = add_gcs(daily, co)
else:
    OBS = list(OBS_FEATURES)
    print("GCS fix off / MIMIC absent — using:", OBS)


✅ GCS_fixed merged (46% of windows)


## Step 1c — Standardise + sequences

In [5]:
# ============================================================
# STEP 1c — ROBUST STANDARDISE + BUILD SEQUENCES
# ============================================================
STD = {}
for c in OBS:
    v = daily[c].dropna().values
    med = float(np.median(v)); iqr = float(np.percentile(v,75)-np.percentile(v,25))
    sd = iqr/1.349 if iqr > 1e-6 else float(np.std(v)); sd = sd if sd > 1e-6 else 1.0
    STD[c] = (med, sd); daily[c] = np.clip((daily[c]-med)/sd, -8, 8)
json.dump(STD, open(OUT/"standardisation.json","w"), indent=2)

seqs = []
for pid, g in daily.groupby("patient_id"):
    O = g[OBS].values.astype(float)
    if len(O) < 2: continue
    seqs.append(dict(pid=pid, O=O, ts=g["day"].values, meld=g["MELD_Na"].values))
print(f"✅ {len(seqs):,} sequences | {sum(len(s['O']) for s in seqs):,} daily windows | D={len(OBS)}")


✅ 7,503 sequences | 75,250 daily windows | D=9


## helper

In [6]:
# helper: attach a chosen treatment key as 'u' for fitting
def seqs_with(seqs, ukey):
    return [dict(O=s["O"], u=s[ukey], pid=s["pid"], meld=s["meld"]) for s in seqs]


## Step 2 — K-sweep

In [7]:
# ============================================================
# STEP 2 — K-SWEEP (restarts; select by predictive LL + monotonicity)
# ============================================================
from scipy.stats import spearmanr

def best_fit(S, K, restarts, n_iter, seed0=1):
    best = None
    for r in range(restarts):
        p = fit(S, K=K, mode="baseline", n_iter=n_iter, seed=seed0+r)
        if best is None or p["loglik"] > best["loglik"]: best = p
    return best

def reorder_meld(p, S):
    K = p["K"]; num = np.zeros(K); den = np.zeros(K)
    for s in S:
        g,_,_ = posterior_stage(p, s["O"]); m = ~np.isnan(s["meld"])
        if m.any(): num += (g[m]*s["meld"][m,None]).sum(0); den += g[m].sum(0)
    order = np.argsort(num/np.maximum(den,1e-6))
    q = dict(p); q["mu"]=p["mu"][order]; q["var"]=p["var"][order]
    q["pi"]=p["pi"][order]/p["pi"][order].sum(); q["A"]=p["A"][np.ix_(order,order)]
    return q

def mortality_by_stage(p, S):
    rows = []
    for s in S:
        _, st, _ = posterior_stage(p, s["O"])
        rows.append((int(round(st.max())), death.get(s["pid"])))
    R = pd.DataFrame(rows, columns=["stage","death"]).dropna()
    return R.groupby("stage")["death"].mean()

def cv_pred(S, K, restarts=2, folds=3, n_iter=15, seed=1):
    rng = np.random.default_rng(seed); fid = np.array_split(rng.permutation(len(S)), folds)
    vals = []
    for f in range(folds):
        te = set(fid[f].tolist()); tr = [S[i] for i in range(len(S)) if i not in te]
        tes = [S[i] for i in fid[f]]
        p = best_fit(tr, K, restarts, n_iter); _,_,per = obs_pred_loglik(p, tes); vals.append(per)
    return float(np.mean(vals)), float(np.std(vals))

print("K | CV pred-LL/win | mortality-by-stage | Spearman(stage,mortality)")
sweep = {}
for K in K_GRID:
    t0 = time.time()
    mu_, sd_ = cv_pred(seqs, K, restarts=2, folds=3, n_iter=15)
    p = reorder_meld(best_fit(seqs, K, K_RESTARTS, N_ITER), seqs)
    mort = mortality_by_stage(p, seqs)
    rho = spearmanr(mort.index, mort.values).correlation if len(mort) > 1 else np.nan
    sweep[K] = dict(cvpred=mu_, cvsd=sd_, spearman=float(rho),
                    mortality=[round(x,3) for x in mort.values])
    print(f"{K} | {mu_:.4f} ± {sd_:.4f} | {sweep[K]['mortality']} | {rho:.3f}  ({time.time()-t0:.0f}s)")
json.dump(sweep, open(OUT/"k_sweep.json","w"), indent=2)

# choose most interpretable K: highest predictive LL among those with monotonic mortality (rho>=0.99)
mono = [K for K in K_GRID if sweep[K]["spearman"] >= 0.99]
BEST_K = (max(mono, key=lambda k: sweep[k]["cvpred"]) if mono
          else max(K_GRID, key=lambda k: sweep[k]["spearman"]))
print(f"\n✅ Selected K = {BEST_K}  (prefers ordinal monotonicity; see table)")


K | CV pred-LL/win | mortality-by-stage | Spearman(stage,mortality)
3 | -9.2169 ± 0.0613 | [np.float64(0.374), np.float64(0.468), np.float64(0.642)] | 1.000  (54s)
4 | -8.7850 ± 0.0658 | [np.float64(0.354), np.float64(0.48), np.float64(0.485), np.float64(0.671)] | 1.000  (77s)
5 | -8.4846 ± 0.1317 | [np.float64(0.346), np.float64(0.47), np.float64(0.492), np.float64(0.462), np.float64(0.707)] | 0.700  (81s)
6 | -8.2853 ± 0.0905 | [np.float64(0.315), np.float64(0.461), np.float64(0.526), np.float64(0.461), np.float64(0.464), np.float64(0.725)] | 0.714  (131s)

✅ Selected K = 4  (prefers ordinal monotonicity; see table)


## Step 3 — Fit best-K staging + validate

In [8]:
# ============================================================
# STEP 3 — FIT BEST-K STAGING + VALIDATE
# ============================================================
stage_params = reorder_meld(best_fit(seqs, BEST_K, K_RESTARTS, N_ITER), seqs)
rows = []
for s in seqs:
    _, st, _ = posterior_stage(stage_params, s["O"])
    rows.append(dict(pid=s["pid"], peak_stage=int(round(st.max())),
                     mean_stage=float(st.mean()), mean_meld=np.nanmean(s["meld"])))
P = pd.DataFrame(rows); P["death"]=P.pid.map(death); P["decomp"]=P.pid.map(decomp)
P.to_csv(OUT/"patient_stages.csv", index=False)
print(f"=== Stage validation (K={BEST_K}, peak stage over stay) ===")
print(P.groupby("peak_stage").agg(n=("pid","size"), mean_MELD=("mean_meld","mean"),
        mortality=("death","mean"), decomp_rate=("decomp","mean")).round(3).to_string())
print("\nTransition matrix (daily, severity-ordered):"); print(np.round(stage_params["A"],4))
np.save(OUT/"A_baseline.npy", stage_params["A"])


=== Stage validation (K=4, peak stage over stay) ===
               n  mean_MELD  mortality  decomp_rate
peak_stage                                         
0           2744     10.985      0.354        0.210
1           1320     15.639      0.480        0.313
2           1796     19.599      0.485        0.478
3           1643     27.240      0.671        0.664

Transition matrix (daily, severity-ordered):
[[9.782e-01 9.600e-03 1.150e-02 7.000e-04]
 [3.770e-02 9.464e-01 5.100e-03 1.080e-02]
 [2.490e-02 5.200e-03 9.497e-01 2.020e-02]
 [2.100e-03 1.220e-02 3.340e-02 9.523e-01]]


## Step 4 — Per-class treatment extraction

In [9]:
# ============================================================
# STEP 4 — PER-CLASS TREATMENT EXTRACTION (daily; needs MIMIC-IV)
# ============================================================
def build_intervals_by_class():
    hosp, icu = os.path.join(MIMIC_ROOT,"hosp"), os.path.join(MIMIC_ROOT,"icu")
    hadm_pid = dict(zip(co.hadm_id, co.patient_id)); hadm_set = set(co.hadm_id.dropna().astype(int))
    cls = {k: {} for k in list(DRUG_REGEX)+list(INPUT_ITEMIDS)+list(PROC_REGEX)}
    pres = _csv(hosp,"prescriptions")
    if pres:
        for ch in pd.read_csv(pres, usecols=["hadm_id","drug","starttime","stoptime"],
                              chunksize=500_000, low_memory=False):
            ch = ch[ch.hadm_id.isin(hadm_set)];
            if ch.empty: continue
            dl = ch["drug"].astype(str).str.lower()
            for name, rx in DRUG_REGEX.items():
                sub = ch[dl.str.contains(rx, regex=True, na=False)]
                for _, r in sub.iterrows():
                    pid = hadm_pid.get(r.hadm_id);  s = pd.to_datetime(r.starttime, errors="coerce")
                    if pid is None or pd.isna(s): continue
                    e = pd.to_datetime(r.stoptime, errors="coerce"); e = e if pd.notna(e) else s+pd.Timedelta("24h")
                    cls[name].setdefault(pid, []).append((s, e))
    inp = _csv(icu,"inputevents")
    if inp:
        for ch in pd.read_csv(inp, usecols=["hadm_id","itemid","starttime","endtime"],
                              chunksize=500_000, low_memory=False):
            ch = ch[ch.hadm_id.isin(hadm_set)]
            if ch.empty: continue
            for name, ids in INPUT_ITEMIDS.items():
                sub = ch[ch.itemid.isin(ids)]
                for _, r in sub.iterrows():
                    pid = hadm_pid.get(r.hadm_id); s = pd.to_datetime(r.starttime, errors="coerce")
                    if pid is None or pd.isna(s): continue
                    e = pd.to_datetime(r.endtime, errors="coerce"); e = e if pd.notna(e) else s+pd.Timedelta("1h")
                    cls[name].setdefault(pid, []).append((s, e))
    proc, di = _csv(icu,"procedureevents"), _csv(icu,"d_items")
    if proc and di:
        d = pd.read_csv(di); lab = "label" if "label" in d.columns else d.columns[-1]
        pid_ids = d[d[lab].astype(str).str.contains(PROC_REGEX["paracentesis"], case=False, na=False)]["itemid"].tolist()
        pe = pd.read_csv(proc, usecols=["hadm_id","itemid","starttime"])
        pe = pe[pe.hadm_id.isin(hadm_set) & pe.itemid.isin(pid_ids)]
        for _, r in pe.iterrows():
            pid = hadm_pid.get(r.hadm_id); s = pd.to_datetime(r.starttime, errors="coerce")
            if pid is None or pd.isna(s): continue
            cls["paracentesis"].setdefault(pid, []).append((s, s+pd.Timedelta("1D")))
    for k in cls: print(f"  {k:<12}: {len(cls[k]):,} patients")
    return cls

def attach_class(seqs, intervals, key):
    w = pd.Timedelta(WINDOW); n = 0
    for s in seqs:
        ts = pd.to_datetime(s["ts"]); u = np.zeros(len(ts))
        for (st, en) in intervals.get(s["pid"], []):
            u[np.asarray((ts < en) & ((ts + w) > st))] = 1.0
        s["u_"+key] = u; n += u.sum()
    tot = sum(len(s["O"]) for s in seqs)
    print(f"  attached '{key}': {n/tot*100:.1f}% windows active")

if os.path.exists(os.path.join(MIMIC_ROOT,"hosp")):
    CLS = build_intervals_by_class()
    for k in CLS: attach_class(seqs, CLS[k], k)
    TREAT_CLASSES = list(CLS)
else:
    TREAT_CLASSES = []
    print("MIMIC-IV not found — Step 5 will be skipped.")


  lactulose   : 3,415 patients
  diuretic    : 4,117 patients
  vasopressor : 1,114 patients
  albumin     : 972 patients
  paracentesis: 233 patients
  attached 'lactulose': 46.2% windows active
  attached 'diuretic': 36.3% windows active
  attached 'vasopressor': 7.6% windows active
  attached 'albumin': 3.9% windows active
  attached 'paracentesis': 0.8% windows active


## Step 5 — Per-class transition-fusion

In [10]:
# ============================================================
# STEP 5 — PER-CLASS TRANSITION-FUSION (fair predictive LL)
# ============================================================
def cv_compare(seqs, K, ukey, n_iter=20, folds=3, seed=RANDOM_SEED):
    S = [dict(O=s["O"], u=s[ukey], pid=s["pid"]) for s in seqs]
    rng = np.random.default_rng(seed); fid = np.array_split(rng.permutation(len(S)), folds)
    out = {m: [] for m in ["baseline","emission","transition"]}
    for f in range(folds):
        te = set(fid[f].tolist()); tr = [S[i] for i in range(len(S)) if i not in te]
        tes = [S[i] for i in fid[f]]
        for mode in out:
            p = fit(tr, K=K, mode=mode, n_iter=n_iter, seed=seed); _,_,per = obs_pred_loglik(p, tes)
            out[mode].append(per)
    return {m: (float(np.mean(v)), float(np.std(v))) for m, v in out.items()}

fusion = {}
for key in TREAT_CLASSES:
    act = np.mean([u for s in seqs for u in s["u_"+key]])
    if act < 0.02 or act > 0.98:
        print(f"[{key}] skipped (prevalence {act*100:.1f}% — too little contrast)"); continue
    res = cv_compare(seqs, BEST_K, "u_"+key, n_iter=20, folds=3)
    fusion[key] = res
    b,e,t = res["baseline"][0], res["emission"][0], res["transition"][0]
    verdict = ("transition" if t > e and t-max(b,e) > res["transition"][1] else
               "emission" if e > t and e-max(b,t) > res["emission"][1] else "no difference")
    print(f"[{key}] prevalence {act*100:.0f}% | baseline {b:.4f} | emission {e:.4f} | "
          f"transition {t:.4f}  -> {verdict}")
    # inspect treatment-conditioned dynamics (severity-ordered via posterior-weighted MELD)
    tpf = fit(seqs_with(seqs, "u_"+key), K=BEST_K, mode="transition", n_iter=N_ITER, seed=RANDOM_SEED)
    K = BEST_K; num = np.zeros(K); den = np.zeros(K)
    for s in seqs:
        g,_,_ = posterior_stage(tpf, s["O"]); m = ~np.isnan(s["meld"])
        if m.any(): num += (g[m]*s["meld"][m,None]).sum(0); den += g[m].sum(0)
    order = np.argsort(num/np.maximum(den,1e-6))
    A_off = tpf["A_off"][np.ix_(order,order)]; A_on = tpf["A_on"][np.ix_(order,order)]
    prog = [(k, round(float(A_off[k,k+1:].sum()),3), round(float(A_on[k,k+1:].sum()),3)) for k in range(K-1)]
    print("   P(progress to higher stage) off/on by stage:", prog)
json.dump({k:{m:list(v) for m,v in r.items()} for k,r in fusion.items()},
          open(OUT/"fusion_per_class.json","w"), indent=2)
print("\n✅ Fusion complete →", OUT/"fusion_per_class.json")


[lactulose] prevalence 46% | baseline -8.7850 | emission -8.7971 | transition -8.7809  -> no difference
   P(progress to higher stage) off/on by stage: [(0, 0.022, 0.022), (1, 0.014, 0.02), (2, 0.017, 0.023)]
[diuretic] prevalence 36% | baseline -8.7850 | emission -8.8740 | transition -8.7857  -> no difference
   P(progress to higher stage) off/on by stage: [(0, 0.02, 0.025), (1, 0.015, 0.018), (2, 0.023, 0.018)]
[vasopressor] prevalence 8% | baseline -8.7850 | emission -8.7754 | transition -8.7840  -> no difference
   P(progress to higher stage) off/on by stage: [(0, 0.021, 0.121), (1, 0.014, 0.036), (2, 0.017, 0.059)]
[albumin] prevalence 4% | baseline -8.7850 | emission -8.7728 | transition -8.7863  -> no difference
   P(progress to higher stage) off/on by stage: [(0, 0.021, 0.094), (1, 0.015, 0.056), (2, 0.018, 0.071)]
[paracentesis] skipped (prevalence 0.8% — too little contrast)

✅ Fusion complete → staging_v2_output/fusion_per_class.json


## Explainability — variance-normalized LLR vs SHAP
Compares the original variance-weighted LLR, a **unit-variance (LDA-style) normalized LLR**, 
and **SHAP** on the advanced-stage attribution, with Spearman rank agreement. Tests whether 
normalizing the LLR for feature scale reconciles it with SHAP.

In [11]:
# ============================================================
# EXPLAINABILITY — LLR (raw + variance-normalized) vs SHAP
# ============================================================
from scipy.stats import spearmanr
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
try:
    import shap as _shap; _HS = True
except Exception:
    _HS = False; print("shap not installed -> exact-linear fallback")

_P = stage_params; _S = seqs; _OBS = list(OBS); _K = BEST_K
_mu, _var = _P["mu"], _P["var"]

def _llr_feat(o, tgt, ref=0, unit=False):
    out = {}
    for j, f in enumerate(_OBS):
        if np.isnan(o[j]): continue
        if unit:   # unit-variance / LDA-style: scale-invariant on standardized obs
            out[f] = (_mu[tgt,j]-_mu[ref,j]) * (o[j]-0.5*(_mu[tgt,j]+_mu[ref,j]))
        else:      # original variance-weighted LLR
            lt = -0.5*(np.log(2*np.pi*_var[tgt,j]) + (o[j]-_mu[tgt,j])**2/_var[tgt,j])
            lr = -0.5*(np.log(2*np.pi*_var[ref,j]) + (o[j]-_mu[ref,j])**2/_var[ref,j])
            out[f] = lt - lr
    return out

_raw = {f: [] for f in _OBS}; _nrm = {f: [] for f in _OBS}; _X = []; _Y = []
for s in _S:
    g, st, _ = posterior_stage(_P, s["O"]); hard = np.argmax(g, axis=1)
    for t in range(len(st)):
        _X.append(s["O"][t]); _Y.append(int(hard[t] == _K-1))
        if hard[t] == _K-1:
            for f, v in _llr_feat(s["O"][t], _K-1, 0, False).items(): _raw[f].append(abs(v))
            for f, v in _llr_feat(s["O"][t], _K-1, 0, True ).items(): _nrm[f].append(abs(v))
Xv = pd.DataFrame(_X, columns=_OBS); Xv = Xv.fillna(Xv.median()); Yv = np.array(_Y)
print(f"visit-level: {len(Xv):,} visits | advanced (top stage) = {int(Yv.sum()):,} ({Yv.mean()*100:.1f}%)")

llr_raw  = pd.Series({f:(np.mean(v) if v else 0.0) for f,v in _raw.items()})
llr_norm = pd.Series({f:(np.mean(v) if v else 0.0) for f,v in _nrm.items()})
_gbm = GradientBoostingClassifier(max_depth=3, n_estimators=200, random_state=0).fit(Xv, Yv)
if _HS:
    _r = _shap.TreeExplainer(_gbm).shap_values(Xv)
    if isinstance(_r, list): _r = _r[-1]
    _r = np.asarray(_r)
    if _r.ndim == 3: _r = _r[:, :, -1]
    shap_imp = pd.Series(np.abs(_r).mean(0), index=_OBS)
else:
    _lin = LogisticRegression(max_iter=2000).fit(Xv, Yv)
    shap_imp = pd.Series(np.abs(_lin.coef_[0])*Xv.std().values, index=_OBS)

cmp = pd.DataFrame({"LLR_raw":llr_raw, "LLR_norm":llr_norm, "SHAP":shap_imp})
for c in ["LLR_raw","LLR_norm","SHAP"]: cmp[c+"_rk"] = cmp[c].rank(ascending=False)
cmp = cmp.sort_values("SHAP", ascending=False)
rho_raw  = spearmanr(cmp["LLR_raw_rk"],  cmp["SHAP_rk"]).correlation
rho_norm = spearmanr(cmp["LLR_norm_rk"], cmp["SHAP_rk"]).correlation
print("\n=== Feature importance: raw-LLR vs norm-LLR vs SHAP (advanced stage) ===")
print(cmp[["LLR_raw","LLR_norm","SHAP","LLR_raw_rk","LLR_norm_rk","SHAP_rk"]].round(3).to_string())
print(f"\nSpearman  SHAP vs raw-LLR  = {rho_raw:+.3f}")
print(f"Spearman  SHAP vs norm-LLR = {rho_norm:+.3f}")
print("  If norm-LLR agrees with SHAP much better, the raw-LLR ranking was a variance artifact.")


visit-level: 75,250 visits | advanced (top stage) = 16,996 (22.6%)

=== Feature importance: raw-LLR vs norm-LLR vs SHAP (advanced stage) ===
            LLR_raw  LLR_norm   SHAP  LLR_raw_rk  LLR_norm_rk  SHAP_rk
Bilirubin   353.871    10.051  2.389         1.0          1.0      1.0
Creatinine   34.449     3.592  1.274         3.0          2.0      2.0
INR          19.283     2.818  0.886         4.0          3.0      3.0
Lactate       5.508     2.098  0.512         6.0          4.0      4.0
GCS_fixed   101.289     1.087  0.126         2.0          6.0      5.0
MAP           0.627     0.676  0.106         7.0          7.0      6.0
Albumin       0.115     0.124  0.082         9.0          9.0      7.0
Ammonia       6.237     1.741  0.076         5.0          5.0      8.0
Sodium        0.618     0.281  0.017         8.0          8.0      9.0

Spearman  SHAP vs raw-LLR  = +0.750
Spearman  SHAP vs norm-LLR = +0.867
  If norm-LLR agrees with SHAP much better, the raw-LLR ranking was a varian